# 17 · Project: from a month of column data to a retrofit proposal

**Problem.** A benzene-toluene column (18 sieve trays, total condenser, kettle reboiler, 1 atm) has produced a
month of daily averages: feed rate and composition, reflux ratio, product purities, temperatures. The plant
wants the distillate at 98.5 % and the bottoms at 1.5 % benzene instead of the current 95/5. The task: clean
the data, determine how many theoretical stages the column actually delivers and its tray efficiency, check
the temperatures against thermodynamics, and decide whether the target can be met by reflux alone, by adding
trays, or by replacing the trays with packing - with a quantified recommendation.

**What you will learn**
- clean plant data and log every repair
- verify thermodynamics against plant temperatures
- back out theoretical stages and tray efficiency from operating data
- evaluate retrofit options quantitatively and write a defensible recommendation

**Before you start:** notebooks 03, 04 and 05  ·  **Time:** about 90 min

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import datasets, vle
from engsep import distillation as dist
from engsep.vle import y_from_alpha
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## 1. Look at the data

In [2]:
p = pd.read_csv(datasets.path("plant_column_benzene_toluene.csv"), comment="#")
print(p.describe().round(3).to_string())
print("\nrows with notes or impossible values:\n", p[(p.note != "ok") | (p.xD > 1) | (p.zF == 0) | (p.T_top_C > 200)].to_string(index=False))

          day  F_kmol_h      zF       R      xD      xB  T_top_C  T_bottom_C
count  31.000    31.000  31.000  31.000  31.000  31.000   31.000      31.000
mean   16.000    99.987   0.435   2.201   3.987   0.048   88.811     104.997
std     9.092     3.074   0.084   0.126  16.904   0.017   51.665      19.499
min     1.000    94.600   0.000   1.980   0.943   0.015    0.000       0.000
25%     8.500    98.000   0.436   2.090   0.948   0.037   82.400     107.900
50%    16.000   100.000   0.452   2.200   0.950   0.048   82.700     108.500
75%    23.500   101.250   0.465   2.300   0.952   0.057   82.950     108.900
max    31.000   106.400   0.481   2.450  95.070   0.095  355.450     109.500

rows with notes or impossible values:
  day  F_kmol_h    zF    R      xD     xB  T_top_C  T_bottom_C                    note
  10      99.9 0.481 2.36 95.0700 0.0258    82.50       109.4                      ok
  17     101.1 0.000 2.09  0.9503 0.0751    82.60       107.4   feed analyser offline
  23     

## 2. Clean, with a log

In [3]:
log, q = [], p[p.note != "target for the retrofit"].copy()
target = p[p.note == "target for the retrofit"].iloc[0]
i = q.index[q.xD > 1]; q.loc[i, "xD"] /= 100; log.append(f"day {int(q.loc[i, 'day'].iloc[0])}: distillate purity was logged in percent")
i = q.index[q.T_top_C > 200]; q.loc[i, "T_top_C"] -= 273.15; log.append(f"day {int(q.loc[i, 'day'].iloc[0])}: top temperature was logged in K")
i = q.index[q.zF == 0]; q = q.drop(i); log.append(f"day {int(p.loc[i, 'day'].iloc[0])}: feed analyser offline, day excluded")
print("\n".join(log)); print(f"\n{len(q)} usable days")

day 10: distillate purity was logged in percent
day 23: top temperature was logged in K
day 17: feed analyser offline, day excluded

29 usable days


## 3. Check the temperatures against thermodynamics

In [4]:
pair = ("benzene", "toluene")
q["T_top_pred"] = [vle.dew_T(pair, xd, ideal=True) - 273.15 for xd in q.xD]
q["T_bot_pred"] = [vle.bubble_T(pair, xb, ideal=True) - 273.15 for xb in q.xB]
dev_top, dev_bot = (q.T_top_C - q.T_top_pred).abs().mean(), (q.T_bottom_C - q.T_bot_pred).abs().mean()
print(f"top: measured {q.T_top_C.mean():.1f} +/- {q.T_top_C.std():.1f} degC vs dew point of the distillate {q.T_top_pred.mean():.1f} (mean |deviation| {dev_top:.2f} K)")
print(f"bottom: measured {q.T_bottom_C.mean():.1f} +/- {q.T_bottom_C.std():.1f} degC vs bubble point of the bottoms {q.T_bot_pred.mean():.1f} (mean |deviation| {dev_bot:.2f} K)")

top: measured 82.7 +/- 0.4 degC vs dew point of the distillate 82.6 (mean |deviation| 0.29 K)
bottom: measured 108.5 +/- 0.7 degC vs bubble point of the bottoms 108.4 (mean |deviation| 0.31 K)


The measured temperatures match the dew point of the distillate and the bubble point of the bottoms within
the thermometers' scatter: the products are what the analysers say, the pressure is 1 atm as recorded, and
Raoult's law with the Antoine table describes the system. Temperatures are the cheapest analyser a column
has, and this check is the first thing to do with them.

## 4. How many theoretical stages does the column deliver?

In [5]:
alpha = vle.geometric_mean_alpha(pair, ideal=True)
y_eq = lambda x: y_from_alpha(x, alpha)
def stages_for(row):
    return dist.mccabe_thiele(y_eq, row.xD, row.xB, row.zF, row.R, q=1.0)["stages"]
q["N_theo"] = q.apply(stages_for, axis=1)
print(q.N_theo.describe().round(2).to_string())
N_theo = int(round(q.N_theo.median()))
E_overall = (N_theo - 1) / 18                                        # stages minus the reboiler, over real trays
print(f"\nthe column delivers about {N_theo} theoretical stages (incl. reboiler): overall tray efficiency {E_overall:.0%}; O'Connell predicts {dist.oconnell_efficiency(alpha, 0.3):.0%}")

count    29.00
mean     11.48
std       0.51
min      11.00
25%      11.00
50%      11.00
75%      12.00
max      12.00

the column delivers about 11 theoretical stages (incl. reboiler): overall tray efficiency 56%; O'Connell predicts 53%


Each day's operating point (feed, reflux, both purities) is a McCabe-Thiele problem whose answer is the number
of theoretical stages the column must have to have produced it; the values cluster around 11 with a
reboiler, so 10 theoretical trays out of 18 real ones - an efficiency of about 55 %, in line with O'Connell's
correlation for this system. The column is neither damaged nor exceptional.

## 5. Can reflux alone reach the target?

In [6]:
zF_avg = q.zF.mean()
rm = dist.minimum_reflux(y_eq, target.xD, zF_avg)
N_min = dist.minimum_stages(y_eq, target.xD, target.xB)
print(f"target 98.5/1.5: minimum stages {N_min} (Fenske {dist.fenske(target.xD, target.xB, alpha):.1f}), minimum reflux {rm['R_min']:.2f}")
rows = []
for R in (2.5, 3.0, 4.0, 6.0, 10.0):
    n = dist.mccabe_thiele(y_eq, target.xD, target.xB, zF_avg, R)["stages"]
    rows.append((R, n, "yes" if n <= N_theo else "no"))
print(pd.DataFrame(rows, columns=["reflux ratio", "theoretical stages needed", f"fits in {N_theo}?"]).to_string(index=False))

target 98.5/1.5: minimum stages 10 (Fenske 9.3), minimum reflux 1.45
 reflux ratio  theoretical stages needed fits in 11?
          2.5                         15          no
          3.0                         14          no
          4.0                         13          no
          6.0                         12          no
         10.0                         11         yes


The target needs more theoretical stages than the column has at any reasonable reflux ratio: only at R = 10,
more than four times the present reboiler duty, does the count fall to the eleven available, with the Fenske
minimum at total reflux just below. Reflux alone would meet the specification at a steam bill no plant would
pay. The column needs more stages - real ones, or better ones.

## 6. Retrofit options

In [7]:
R_design = 1.3 * rm["R_min"]
N_needed = dist.mccabe_thiele(y_eq, target.xD, target.xB, zF_avg, R_design)["stages"]
print(f"at R = 1.3 R_min = {R_design:.2f}: {N_needed} theoretical stages needed")
options = [
    ("more trays, same efficiency", int(np.ceil((N_needed - 1) / E_overall)), f"{int(np.ceil((N_needed - 1) / E_overall)) - 18} extra trays: a taller shell"),
    ("high-efficiency trays (E = 0.75)", int(np.ceil((N_needed - 1) / 0.75)), "replace tray internals in the existing shell if it fits"),
    ("structured packing (HETP 0.4 m)", None, f"{dist.packed_height(N_needed - 1, 0.4):.1f} m of packing"),
]
for name, n_trays, note in options:
    print(f"{name:<34}: " + (f"{n_trays} trays; " if n_trays else "") + note)
print(f"\nexisting shell: 18 trays at 0.6 m spacing = 10.8 m; the retrofit must fit in it or the shell grows")

at R = 1.3 R_min = 1.88: 19 theoretical stages needed
more trays, same efficiency       : 33 trays; 15 extra trays: a taller shell
high-efficiency trays (E = 0.75)  : 24 trays; replace tray internals in the existing shell if it fits
structured packing (HETP 0.4 m)   : 7.2 m of packing

existing shell: 18 trays at 0.6 m spacing = 10.8 m; the retrofit must fit in it or the shell grows


Three ways to get the missing stages: a taller column with more of the same trays (a new shell), better
trays in the same shell (high-efficiency valve trays can reach 75 %), or structured packing, whose height
equivalent to a theoretical plate of 0.4 m puts the needed stages in less height than the trays occupy now.
Packing in the existing shell is the usual retrofit answer, and the calculation says it fits with room to
spare.

## 7. The report

In [8]:
report = f'''# Benzene-toluene column: retrofit basis

Data: 30 daily averages; {'; '.join(log)}. {len(q)} days used.
Thermodynamics: Raoult's law with the Antoine table; measured top/bottom temperatures agree with the product dew/bubble points to {dev_top:.1f} and {dev_bot:.1f} K on average.
Column performance: {N_theo} theoretical stages incl. reboiler from McCabe-Thiele on each day's operating point (median; interquartile range {q.N_theo.quantile(0.25):.0f}-{q.N_theo.quantile(0.75):.0f}); overall efficiency {E_overall:.0%} (O'Connell {dist.oconnell_efficiency(alpha, 0.3):.0%}).
Target 98.5 / 1.5 mol % benzene: N_min = {N_min}, R_min = {rm['R_min']:.2f}; not reachable by reflux with the present stages.
Recommendation: replace the trays with structured packing (HETP 0.4 m), {dist.packed_height(N_needed - 1, 0.4):.1f} m for {N_needed - 1} theoretical stages, operated at R = {R_design:.2f} ({(R_design + 1)/(2.2 + 1):.2f} x the present vapour rate and reboiler duty); fits the existing 10.8 m shell.
Checks before commitment: vendor HETP at the design vapour load, flooding at R = {R_design:.2f} (notebook 05), and a rigorous simulation with non-constant relative volatility.
'''
print(report)

# Benzene-toluene column: retrofit basis

Data: 30 daily averages; day 10: distillate purity was logged in percent; day 23: top temperature was logged in K; day 17: feed analyser offline, day excluded. 29 days used.
Thermodynamics: Raoult's law with the Antoine table; measured top/bottom temperatures agree with the product dew/bubble points to 0.3 and 0.3 K on average.
Column performance: 11 theoretical stages incl. reboiler from McCabe-Thiele on each day's operating point (median; interquartile range 11-12); overall efficiency 56% (O'Connell 53%).
Target 98.5 / 1.5 mol % benzene: N_min = 10, R_min = 1.45; not reachable by reflux with the present stages.
Recommendation: replace the trays with structured packing (HETP 0.4 m), 7.2 m for 18 theoretical stages, operated at R = 1.88 (0.90 x the present vapour rate and reboiler duty); fits the existing 10.8 m shell.
Checks before commitment: vendor HETP at the design vapour load, flooding at R = 1.88 (notebook 05), and a rigorous simulation 

Every number in the report traces to a file, a method and a check; the unit faults are logged, the
thermodynamics were verified against the plant's own thermometers, and the recommendation is quantitative
with its remaining uncertainties named. The tools were McCabe-Thiele, Fenske, O'Connell and an HETP - the
content of notebooks 03 to 05, applied to real-looking data.

## Inside the algorithm: stages from an operating point

In [9]:
row_ = q.iloc[0]
print("day", int(row_.day), ": xD", row_.xD, "xB", row_.xB, "zF", row_.zF, "R", row_.R, "->", dist.mccabe_thiele(y_eq, row_.xD, row_.xB, row_.zF, row_.R)["stages"], "theoretical stages")

day 1 : xD 0.9475 xB 0.0383 zF 0.459 R 2.25 -> 11 theoretical stages


## Exercises
1. The plant also asks whether the *feed tray* is in the right place. For the current operation, find the optimum feed stage from McCabe-Thiele and compare with the actual feed tray (tray 9 from the top). How many stages are lost if the feed is two trays too high?
2. Hydraulics: at the retrofit reflux ratio the vapour rate rises. Using the flooding estimate of notebook 05 (vapour density 2.8 kg/m3, liquid 800, surface tension 0.021 N/m) and a 1.2 m shell, check that the packing operates below 80 % of flooding. If not, what reflux ratio is the hydraulic limit, and how many stages does that require?
3. **Implement it yourself:** write `column_stages(df)` that returns the theoretical stage count for every day and its median, and confirm it reproduces the notebook's table.